<a href="https://colab.research.google.com/github/sana200420/naari-ai/blob/sana%2Ftest-protection/retrieval/scripts/link_gold_queries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Link Gold_KB_final_280 to answer_id

`eval/gold_kb_final_280_raw.csv` has 280 real colloquial-Sindhi questions with their own written answers and citations, but **no `correct_answer_id`** -- it can't be used to measure Recall@K without one.

This embeds each of the 280 questions with bge-m3 and queries them against the **already-built** `naari_ai_kb` Qdrant collection (dense search) to find the best-matching existing KB row. Every match is recorded with its score and top-3 alternatives so low-confidence links can be spotted and reviewed by a person, not silently trusted.

**Important limitation, read before treating the output as final:** this uses the same embedding model as the retrieval system being evaluated. That's fine for *linking* a question to its answer (record linkage), but it does not fix the deeper concern flagged in PLAYBOOKS.md -- these 280 questions still don't look like they were drawn from real harvested speech (Lever 2's seed_real.csv). Getting `correct_answer_id` filled in makes this file *usable*; it doesn't by itself make it *independent*. Flag that back to whoever produced it.

Needs the same Qdrant credentials as before, and **Runtime -> Change runtime type -> T4 GPU**.

In [10]:
!pip install -q qdrant-client FlagEmbedding

In [11]:
!rm -rf naari-ai
!git clone --branch sana/test-protection --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import sys
sys.path.insert(0, ".")

from retrieval.normalize import normalize_sd
from retrieval.search import HybridRetriever

print("cloned + imported OK")

Cloning into 'naari-ai'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 62 (delta 4), reused 49 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 525.93 KiB | 1.01 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/naari-ai/naari-ai
cloned + imported OK


In [12]:
import csv

with open("eval/gold_kb_final_280_raw.csv", encoding="utf-8-sig", newline="") as f:
    gold_rows = list(csv.DictReader(f))

print(f"{len(gold_rows)} gold candidate rows loaded")
print(gold_rows[0])

280 gold candidate rows loaded
{'ID': '1', 'Category': 'Pregnancy & Maternal Health', 'Subcategory': 'Early Pregnancy', 'Question': 'منهنجي ماهواري رهجي وئي آهي ۽ مون کي ڪمزوري به محسوس ٿئي ٿي، ڇا مان حامله ٿي سگهان ٿي؟', 'Answer': 'ماهواري رهجي وڃڻ سان گڏ ڪجهه ٻين نشانين جي موجودگي حمل جي نشاني ٿي سگهي ٿي، پر پڪ لاءِ حمل جو ٽيسٽ ڪرڻ گهرجي.', 'Source': 'World Health Organization (2016). WHO recommendations on antenatal care for a positive pregnancy experience. Geneva: WHO', 'Evidence_Type': 'Clinical guideline', 'Supported_Claim': 'Guideline covers early confirmation of pregnancy and prompt initiation of antenatal care; it does not evaluate symptom-based diagnosis.', 'Claim_Support': 'General guidance'}


In [13]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("model loaded")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model loaded


In [14]:
from getpass import getpass
from qdrant_client import QdrantClient

QDRANT_URL = getpass("Qdrant cluster URL: ")
QDRANT_API_KEY = getpass("Qdrant API key: ")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def embed_fn(text):
    normalized = normalize_sd(text)
    out = model.encode([normalized], return_dense=True, return_sparse=True, return_colbert_vecs=False)
    return {"dense": out["dense_vecs"][0].tolist(), "sparse": out["lexical_weights"][0]}

retriever = HybridRetriever(client, embed_fn=embed_fn)
print("retriever ready, collection points:", client.get_collection("naari_ai_kb").points_count)

Qdrant cluster URL: ··········
Qdrant API key: ··········
retriever ready, collection points: 2000


In [15]:
linked = []

for i, row in enumerate(gold_rows, start=1):
    query = row["Question"]
    hits = retriever.dense_search(query, top_k=3)

    top1 = hits[0] if len(hits) > 0 else None
    top2 = hits[1] if len(hits) > 1 else None
    top3 = hits[2] if len(hits) > 2 else None

    linked.append({
        "query_id": f"gold_{row['ID']}",
        "query": query,
        "stated_category": row["Category"],
        "stated_subcategory": row["Subcategory"],
        "correct_answer_id": top1["answer_id"] if top1 else "",
        "top1_score": f"{top1['score']:.4f}" if top1 else "",
        "top1_category": top1["category"] if top1 else "",
        "category_match": (top1["category"] == row["Category"]) if top1 else False,
        "top2_answer_id": top2["answer_id"] if top2 else "",
        "top2_score": f"{top2['score']:.4f}" if top2 else "",
        "top3_answer_id": top3["answer_id"] if top3 else "",
        "top3_score": f"{top3['score']:.4f}" if top3 else "",
        "gold_own_answer": row["Answer"],
        "gold_own_source": row["Source"],
    })

    if i % 40 == 0:
        print(f"linked {i}/{len(gold_rows)}")

print(f"done, {len(linked)} rows linked")

linked 40/280
linked 80/280
linked 120/280
linked 160/280
linked 200/280
linked 240/280
linked 280/280
done, 280 rows linked


In [16]:
import csv

OUT_PATH = "eval/gold_eval_280_linked.csv"
fieldnames = list(linked[0].keys())
with open(OUT_PATH, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(linked)

print(f"wrote {OUT_PATH}")

mismatches = [r for r in linked if not r["category_match"]]
low_conf = [r for r in linked if r["top1_score"] and float(r["top1_score"]) < 0.6]
print(f"category mismatches (top-1 KB category != gold's stated category): {len(mismatches)}/{len(linked)}")
print(f"low-confidence matches (top1_score < 0.6): {len(low_conf)}/{len(linked)}")
print("-> both of these groups need a human to actually look at them before trusting the link.")

wrote eval/gold_eval_280_linked.csv
category mismatches (top-1 KB category != gold's stated category): 280/280
low-confidence matches (top1_score < 0.6): 2/280
-> both of these groups need a human to actually look at them before trusting the link.


## Next steps

1. Download `eval/gold_eval_280_linked.csv` from the Colab file browser (or commit it back to the repo).
2. **Someone reviews every row flagged by `category_match = False` or `top1_score < 0.6`** -- these are the links most likely to be wrong. For a wrong link, either correct `correct_answer_id` by hand (look at `top2`/`top3` as candidates, or search the KB directly) or drop the row if nothing in the KB actually covers it yet.
3. Once reviewed, this becomes the real `eval/gold_eval.csv` -- but see the limitation note at the top: getting IDs filled in does not by itself resolve whether these 280 questions were independently sourced. Confirm that separately with whoever built this file.
4. Only after review should this feed Sana's dense-only baseline (Recall@1/5/20) and the Lever 3 ablation table.

In [17]:
import csv

# top1["category"] comes back in Sindhi; row["Category"] (stated_category) is
# English. Comparing them directly always fails -- that's the 280/280 bug.
# Translate before comparing, using the KB's own category order.
with open("knowledge_base/Womens_Health_KB - 2000_final.csv", encoding="utf-8", newline="") as f:
    kb_rows_for_mapping = list(csv.DictReader(f))

sindhi_cats_order = []
seen_cats = set()
for r in kb_rows_for_mapping:
    if r["category"] not in seen_cats:
        sindhi_cats_order.append(r["category"])
        seen_cats.add(r["category"])

EN_CATEGORY_ORDER = [
    "Menstrual Health & Periods",
    "Mental Health & Emotional Well-being",
    "Pregnancy & Maternal Health",
    "PCOS & Hormonal Health",
    "Women's Nutrition & Wellness",
    "Menopause & Menopausal Health",
    "Fertility & Reproductive Health",
    "Vaginal & Personal Hygiene",
]
sd_to_en_category = dict(zip(sindhi_cats_order, EN_CATEGORY_ORDER))

for r in linked:
    r["top1_category_en"] = sd_to_en_category.get(r["top1_category"], r["top1_category"])
    r["category_match"] = (r["top1_category_en"] == r["stated_category"])

OUT_PATH = "eval/gold_eval_280_linked.csv"
fieldnames = list(linked[0].keys())
with open(OUT_PATH, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(linked)

print(f"wrote {OUT_PATH}")

mismatches = [r for r in linked if not r["category_match"]]
low_conf = [r for r in linked if r["top1_score"] and float(r["top1_score"]) < 0.6]
print(f"category mismatches (now correctly translated): {len(mismatches)}/{len(linked)}")
print(f"low-confidence matches (top1_score < 0.6): {len(low_conf)}/{len(linked)}")


wrote eval/gold_eval_280_linked.csv
category mismatches (now correctly translated): 146/280
low-confidence matches (top1_score < 0.6): 2/280
